In [ ]:
import mass
import numpy as np
import matplotlib.pyplot as plt
import glob

import sys
sys.path.append('../Utils/')
from fixed_autocuts import auto_cuts 
from roosa_utils import *

For the PAX 2026 Zr run we calibrated with 57Co. 
This produced escape peaks in the tin absorbers which were useful "extra" calibration points. 

In [ ]:
g_57Co = [122.06065e3, 136.47356e3]

x_Sn    = [25.2713e3, 25.0440e3, 28.4860e3]
g_241Am = [59.5409e3, 26.3446e3]
x_241Am = [] 
g_133Ba = [356.0129e3, 302.8508e3, 276.3989e3, 223.2368e3, 160.6120e3, 80.9979e3, 79.6142e3, 53.1622e3]
x_133Ba = [35.818e3, 34.987e3, 34.92e3, 30.973e3, 30.625e3]

When we have many runs, it can be nice to have an indexed list like this to organize the addresses of pulse/noise files 

In [ ]:
pulse_files = {}
noise_files = {}

dir_data = "/Volumes/PAX2026"

noise_files[22] = "/data/20260228/0001/20260228_run0001_chan*.ljh"
pulse_files[22] = "/data/20260228/0002/20260228_run0002_chan*.ljh"

In [ ]:
run_no = 22
outf_name= "run"+str(run_no)+".csv"

pulsePattern = sorted(glob.glob(dir_data+pulse_files[run_no]))
noisePattern = sorted(glob.glob(dir_data+noise_files[run_no]))

data:mass.TESGroup = mass.TESGroup(pulsePattern, noisePattern, overwrite_hdf5_file=True)

In [ ]:
#apply base MASS analysis algorithms 
data.compute_noise(forceNew=False)#True)
data.summarize_data(forceNew=False)#True)

In [ ]:
data.correct_flux_jumps(4000)

In [ ]:
#apply re-worked autocuts
for ds in data: ds.clear_cuts()
for ds in data: auto_cuts(ds,clearCuts=True, forceNew=True)

In [ ]:
test_chan = 9
fig=plt.figure(figsize=(14,18))
data.channel[test_chan].plot_summaries()

plt.suptitle("run "+str(run_no)+" channel "+str(test_chan)+" summary", fontsize=16)
plt.savefig("run"+str(run_no)+"ch"+str(test_chan)+"_summaries.png")

Pause here and look at the pre-trigger mean for the run. Try to guess where the heater-out dropped below 12% or wherever the funny stuff starts. 

The flux jump corrector has a hard time fixing all the thermometry jumps. A fast fix is to just cut the uncorrected bits. 

In [ ]:

#log the time cuts manually
timecut_sec={}

timecut_sec[22] = (data.channel[9].p_timestamp[0]+0*60,   data.channel[9].p_timestamp[0]+1075*60)

#apply the cut
timestamp_cuts = mass.controller.AnalysisControl(
    timestamp_sec=timecut_sec[run_no]  # INCLUDES pulses from -inf to unix timestamp 1771902941
)


#print('Applying timestamp cuts')
for ds in data.datasets: ds.apply_cuts(timestamp_cuts)


Check if there are any stray PTM jumps 

In [ ]:
chans = data.good_channels # all data
for nn,ch in enumerate(chans): check_ptm(data, ch, run_no)

Scale the pulse integral (average) to energy for a coarse calibration. 
From here we can make a rough energy cut before building the optimal filter

In [ ]:
emin = g_57Co[0]-5e3
emax = g_57Co[0]+5e3
nbins_e = int((emax-emin)/(20))   #50 eV bins           

chans = data.good_channels # all data
all_histos_E = np.zeros((2, len(chans), nbins_e))
coaddE = np.zeros((2, nbins_e))


for nn,ch in enumerate(chans): 
    ds = data.channel[ch]
    g = ds.good()

    
    ds.p_pulse_average = mode_norm(ds.p_pulse_average, g, no_bins = 8000, search_range = [0, 8000], norm = g_57Co[0])
    #print(ch, pavg_mode)
    
    
    #emin *= g_57Co[0]/pavg_mode
    #emax *= g_57Co[0]/pavg_mode
    
    energy_range = (emin, emax)
    #print(energy_range)
    
    hE, bE  = np.histogram(ds.p_pulse_average, bins=nbins_e, range=energy_range)
    hEg, bEg = np.histogram(ds.p_pulse_average[g], bins=nbins_e, range=energy_range)
    all_histos_E[0,nn,:] = hE
    all_histos_E[1,nn,:] = hEg
    coaddE[0,:] += hE
    coaddE[1,:] += hEg 
    
    ebins = bE

plt.figure(figsize=(14,4))
plt.semilogy(ebins[:-1],coaddE[0,:], label = 'all events')
plt.semilogy(ebins[:-1],coaddE[1,:], label = 'all good')

maxval = hE.max() 
    

plt.xlim((emin, emax))
plt.xlabel(('energy (eV)'))
plt.legend()
plt.title("run"+str(run_no)+' coadded pulse avg spectrum')
#plt.savefig("run"+str(run_no)+"coadd_spectrum_nominal.png")




Make a histo of the risetimes and find the most likely value. 
Divide this out for a mode-normalized rise-time

In [ ]:
#regularize the rise times 
for ch in data.good_channels:
    ds = data.channel[ch]
    g = ds.good()
 
    ds.p_rise_time[:] = mode_norm(ds.p_rise_time, g, no_bins=5000, search_range=(0.0005,.001))

In [ ]:
#regularize the rise times 
plt.figure(figsize=(16,3))
for ch in data.good_channels:
    ds = data.channel[9]
    g = ds.good()

    hRT, bRT  = np.histogram(ds.p_rise_time, bins=5000, range = (.75,1.1))

    plt.semilogy(bRT[:-1],hRT[:], label = 'all events')

plt.title("run "+str(run_no)+" Rise Times")
plt.xlabel(('Risetime (Frac. of Max by chan)'))
plt.ylabel(('Counts'))
    #plt.legend()
plt.savefig("rts"+str(run_no)+".png")

In [ ]:
half_window = 250

mask_min = g_57Co[0]-half_window
mask_max = g_57Co[0]+half_window

masks = data.make_masks(pulse_avg_range=[mask_min,mask_max])

for nn,ch in enumerate(chans): 
    ds = data.channel[ch]
    g = ds.good()
    mask_temp =  np.logical_and(ds.p_rise_time[:]<1.01, ds.p_rise_time[:]>.98)
    masks[nn] = np.logical_and(masks[nn], mask_temp)

Make the average pulses and calculate the optimal filter 

In [ ]:
#calculate filters 
data.compute_average_pulse(masks) 
#data.avg_pulses_auto_masks(forceNew=True)#also very suspiscious 

data.compute_5lag_filter(f_3db=500,forceNew=True)
data.filter_data(forceNew=True)

In [ ]:
#generate predicted e resolution 
for nn,ch in enumerate(chans): 
    print("channel "+str(ch))
    data.channel[ch].filter.report(g_57Co[0])
    print("")
    #print(ch, ds.filter.predicted_v_over_dv)

In [ ]:
#apply drift correction
data.drift_correct(forceNew=True)
data.phase_correct(forceNew=True) #highly suspicious -> maybe extra 

chans = data.good_channels # all data
for nn,ch in enumerate(chans): check_dc(data, ch, run_no)

In [ ]:
#applys the dc style entropy minimizer to the median filt value over time
data.time_drift_correct(forceNew=True)

In [ ]:
my_cal = []#[93.574e3, 97.016e3, 122.06065e3, 136.47356e3]

my_cal.extend(g_57Co)
my_cal.extend(give_escapes(g_57Co, x_Sn[0:1]))

my_cal.sort()
print(my_cal)

for i,ds in enumerate(data):
    #g = ds.good()
    print('Working on channel %d'%ds.channum)
    
    try:
        #cal initial
        ds.calibrate('p_filt_value_tdc',line_names=my_cal,forceNew=True)
    except:
        print('failed to calibrate ch '+str(ds.channum))

In [ ]:
###### plot noise
plt.figure(figsize=(5,4))
data.plot_noise(legend=False)
plt.title("run"+str(run_no)+'noise spectrum')
plt.savefig("run"+str(run_no)+"_noise.png")

In [ ]:
''' 
Energy Histogram
0: all events 
1: all good events
'''

energy_range = (0,350e3) #0 - 350keV
nbins_e = int((350e3)/(50))   #50 eV bins           

chans = data.good_channels # all data
all_histos_E = np.zeros((2, len(chans), nbins_e))
coaddE = np.zeros((2, nbins_e))

for nn,ch in enumerate(chans): 
    ds = data.channel[ch]
    g = ds.good()
    
    hE, bE  = np.histogram(ds.p_energy[:], bins=nbins_e, range=energy_range)
    hEg, bEg = np.histogram(ds.p_energy[g], bins=nbins_e, range=energy_range)
    all_histos_E[0,nn,:] = hE
    all_histos_E[1,nn,:] = hEg
    coaddE[0,:] += hE
    coaddE[1,:] += hEg 
    
ebins = bE

In [ ]:
chans = data.good_channels # all data
plt.figure(figsize=(15,9))
for nn, ch in enumerate(chans):
    
    #plt.semilogy(ebins[:-1],all_histos_E[0,nn,:]+nn, label = 'all events')
    #calculate a log scale offset by chan
    #offset = 1000**(float(ch)/128)
    #calculate a lin scale offset by chan
    offset = 10**ch
    plt.plot(ebins[:-1], (all_histos_E[1,nn,:]+1)*offset, label = 'all good')

plt.yscale("log")
plt.xlim((1e3, 380e3))
plt.xlabel(('energy (eV)'))
#plt.legend()
plt.title('offset energy spectra by channel')   

In [ ]:
plt.figure(figsize=(14,4))
plt.semilogy(ebins[:-1],coaddE[0,:], label = 'all events')
plt.semilogy(ebins[:-1],coaddE[1,:], label = 'all good')

maxval = hE.max() 

for line in my_cal: 
    plt.plot([line,line],[0, maxval*1e2])
    

plt.xlim((1e3, 200e3))
plt.xlabel(('energy (eV)'))
plt.legend()
plt.title("run"+str(run_no)+' coadded energy spectrum')
plt.savefig("run"+str(run_no)+"coadd_spectrum_nominal.png")

In [ ]:
#Calculate the time since trigger 
dt_trig = {} #for time since nearest preceding trigger 
id_trig = {} #index for above

for ch in data.good_channels: #loop channels
    ds = data.channel[ch] 
    dt_trig_ch = [] #by chan place holder
    id_trig_ch = [] #by chan place holder 
    for i in range(ds.nPulses): #loop pulses 
        #make array of 
        t_from_trigger = np.array(ds.p_subframecount[i]-data.external_trigger_subframe_count[:])
        #t_from_trigger = t_from_trigger[t_from_trigger>0]
        min_time_arg = np.abs(t_from_trigger).argmin()
        dt_trig_ch.append(t_from_trigger[min_time_arg])
        id_trig_ch.append(min_time_arg)
    dt_trig[ch]= np.asarray(dt_trig_ch)*data.subframe_timebase*1000 #ms ? 
    id_trig[ch]= np.asarray(id_trig_ch)

In [ ]:
''' 
Prompt Energy Histogram
0: all events 
1: all good events
'''

energy_range = (0,350e3) #0 - 350keV
nbins_e = int((350e3)/(50))   #50 eV bins           

chans = data.good_channels # all data
all_histos_EP = np.zeros((2, len(chans), nbins_e))
coaddEP = np.zeros((2, nbins_e))

for nn,ch in enumerate(chans): 
    #if ch !=21: continue
    ds = data.channel[ch]
    g = ds.good()
    prompt = np.logical_and(dt_trig[ch]>1.03, dt_trig[ch]<1.07)
    good_risetime = ds.p_rise_time[:]>.95
    gprompt = np.logical_and(g, prompt, good_risetime)
    
    hEP, bEP   = np.histogram(ds.p_energy[prompt],  bins=nbins_e, range=energy_range)
    hEPg, bEPg = np.histogram(ds.p_energy[gprompt], bins=nbins_e, range=energy_range)
    all_histos_EP[0,nn,:] = hEP
    all_histos_EP[1,nn,:] = hEPg
    coaddEP[0,:] += hEP
    coaddEP[1,:] += hEPg 
    
ebins = bEP

In [ ]:
plt.figure(figsize=(14,4))
plt.semilogy(ebins[:-1],coaddEP[0,:], label = 'all events')
plt.semilogy(ebins[:-1],coaddEP[1,:], label = 'all good')

#maxval = hEP.max() 

plt.xlim((1e3, 200e3))
plt.xlabel(('energy (eV)'))
plt.legend()
plt.title("run"+str(run_no)+' coadded prompt energy spectrum')
plt.savefig("run"+str(run_no)+"coadd_prompt_spectrum_nominal.png")

In [ ]:
#calulate the time between each trigger
trigger_times = data.external_trigger_subframe_count[:]*data.subframe_timebase * 1000
spill_id, mb_id = calc_spill_mb_no(trigger_times, 110) #spill threshold in ms

#Finally, make the coadded 
#recall
#dt_trig = {} #for time since nearest preceding trigger 
#id_trig = {} #index for above
coadded_mb_id = np.array([])
coadded_sp_id = np.array([])

for ch in data.good_channels:
    coadded_mb_id = np.append(coadded_mb_id, mb_id[id_trig[ch]])
    coadded_sp_id = np.append(coadded_sp_id, spill_id[id_trig[ch]])
    
    if len(mb_id[id_trig[ch]])!=len(data.channel[ch].p_filt_value): print("MB and Spill assignment failed for ch ",ch)